# 01 - Full Feature Test Notebook

This notebook is meant to test the full video assistant workflow on the videos in `experiments/videos`.

It covers every major feature currently implemented:

- path and environment setup
- runtime configuration
- video indexing
- transcript extraction when audio is available
- frame-based visual indexing
- summary generation
- index persistence and reload
- transcript-oriented search
- visual timestamp search
- question answering over one indexed video
- manual search / QA playground

The notebook is designed for real runs, not mocks. It expects:

- `uv sync` already done
- `ffmpeg` available on `PATH`
- `NVIDIA_API_KEY` exported in the environment

## 1. Resolve project paths

This cell makes the notebook portable whether Jupyter starts from the repo root or from inside the notebook folder.

In [1]:
from pathlib import Path
import sys


def find_repo_root(start: Path) -> Path:
    for candidate in [start, *start.parents]:
        if (candidate / "src").exists() and (candidate / "experiments" / "videos").exists():
            return candidate
    raise RuntimeError("Could not locate the repository root from the current working directory.")


REPO_ROOT = find_repo_root(Path.cwd().resolve())
SRC_DIR = REPO_ROOT / "src"
VIDEOS_DIR = REPO_ROOT / "experiments" / "videos"
NOTEBOOK_DIR = REPO_ROOT / "experiments" / "notebook"
ARTIFACTS_DIR = NOTEBOOK_DIR / "artifacts"
INDEX_DIR = ARTIFACTS_DIR / "indexes"
WORK_DIR = ARTIFACTS_DIR / "work"

INDEX_DIR.mkdir(parents=True, exist_ok=True)
WORK_DIR.mkdir(parents=True, exist_ok=True)

if str(SRC_DIR) not in sys.path:
    sys.path.insert(0, str(SRC_DIR))

print(f"REPO_ROOT  : {REPO_ROOT}")
print(f"VIDEOS_DIR : {VIDEOS_DIR}")
print(f"INDEX_DIR  : {INDEX_DIR}")

REPO_ROOT  : C:\Users\prxch\Desktop\VideoSearch
VIDEOS_DIR : C:\Users\prxch\Desktop\VideoSearch\experiments\videos
INDEX_DIR  : C:\Users\prxch\Desktop\VideoSearch\experiments\notebook\artifacts\indexes


## 2. Import the project and check prerequisites

This notebook uses the real runtime. If this cell fails, fix the environment first.

In [2]:
import json
import os
import shutil

from video_search_tool.cli import build_assistant_runtime
from video_search_tool.config import AppConfig, SearchConfig
from video_search_tool.models import VideoIndex

ffmpeg_path = shutil.which("ffmpeg")
api_key = os.getenv("NVIDIA_API_KEY", "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").strip()

if not ffmpeg_path:
    raise RuntimeError("ffmpeg is not available on PATH.")
if not api_key:
    raise RuntimeError("NVIDIA_API_KEY is not set in the environment.")

print(f"ffmpeg: {ffmpeg_path}")
print("NVIDIA_API_KEY: present")

ffmpeg: C:\Users\prxch\AppData\Local\Microsoft\WinGet\Packages\Gyan.FFmpeg_Microsoft.Winget.Source_8wekyb3d8bbwe\ffmpeg-8.1-full_build\bin\ffmpeg.EXE
NVIDIA_API_KEY: present


## 3. Discover the videos and define the test plan

The notebook targets every `.mp4` file in `experiments/videos`.

You can also customize the visual test query and QA question for each video.

In [3]:
video_paths = sorted(VIDEOS_DIR.glob("*.mp4"))
assert video_paths, f"No .mp4 files found in {VIDEOS_DIR}"

for index, video_path in enumerate(video_paths, start=1):
    print(f"{index}. {video_path.name} ({video_path.stat().st_size / 1024:.1f} KB)")

VIDEO_TEST_CASES = {
    "paris": {
        "visual_query": "Arc de Triomphe",
        "question": "When does the Arc de Triomphe appear?"
    },
    "gazelle": {
        "visual_query": "gazelle",
        "question": "When does the gazelle appear?"
    },
    "waterfall": {
        "visual_query": "waterfall",
        "question": "When do we see the waterfall?"
    },
}

DEFAULT_TARGET_VIDEOS = [video_path.name for video_path in video_paths]
DEFAULT_TARGET_VIDEOS

1. gazelle.mp4 (5362.6 KB)
2. paris.mp4 (4369.5 KB)
3. waterfall.mp4 (625.3 KB)


['gazelle.mp4', 'paris.mp4', 'waterfall.mp4']

## 4. Notebook settings

Edit these values if needed, then rerun this cell.

In [4]:
TARGET_VIDEOS = DEFAULT_TARGET_VIDEOS.copy()
FORCE_REINDEX = True
TOP_K = 5

config = AppConfig(
    search=SearchConfig(
        candidate_pool_size=12,
        result_limit=TOP_K,
        min_semantic_score=0.0,
        reranking_enabled=True,
        answer_evidence_count=6,
    )
)

print("TARGET_VIDEOS =", TARGET_VIDEOS)
print("FORCE_REINDEX =", FORCE_REINDEX)
print("TOP_K =", TOP_K)

TARGET_VIDEOS = ['gazelle.mp4', 'paris.mp4', 'waterfall.mp4']
FORCE_REINDEX = True
TOP_K = 5


## 5. Build the runtime and helper functions

This creates the indexer, searcher, assistant, and store used by the rest of the notebook.

In [5]:
indexer, searcher, assistant, store = build_assistant_runtime(config)


def index_output_path(video_path: Path) -> Path:
    return INDEX_DIR / f"{video_path.stem}.index.json"


def load_or_build_index(video_path: Path, force_reindex: bool = False) -> VideoIndex:
    output_path = index_output_path(video_path)
    if force_reindex or not output_path.exists():
        print(f"Indexing {video_path.name} -> {output_path.name}")
        index = indexer.index_video(
            video_path,
            video_id=video_path.stem,
            metadata={
                "filename": video_path.name,
                "source": "experiments/videos",
            },
            working_directory=WORK_DIR / video_path.stem,
        )
        store.save(index, output_path)
    else:
        print(f"Loading cached index for {video_path.name}")
    return store.load(output_path)


def first_audio_query(index: VideoIndex) -> str | None:
    for segment in index.transcript_segments:
        text = segment.text.strip()
        if text:
            words = text.split()
            return " ".join(words[: min(6, len(words))])
    return None


def result_to_dict(result):
    return {
        "rank": result.rank,
        "modality": result.modality,
        "label": result.label,
        "time_range": f"{result.start_seconds:.2f}s -> {result.end_seconds:.2f}s",
        "semantic_score": round(result.semantic_score, 4),
        "rerank_score": None if result.rerank_score is None else round(result.rerank_score, 4),
        "final_score": round(result.final_score, 4),
        "text": result.text,
    }


## 6. Build or load all selected indexes

This is the main indexing step. It will:

- extract audio when available
- transcribe speech
- extract video frames
- caption visual frames
- embed searchable evidence
- generate a video summary
- save the JSON index

If `FORCE_REINDEX = False`, cached indexes are reused.

In [6]:
selected_video_paths = [path for path in video_paths if path.name in TARGET_VIDEOS]
assert selected_video_paths, "TARGET_VIDEOS does not match any available video."

indexes_by_video = {}
for video_path in selected_video_paths:
    indexes_by_video[video_path.stem] = load_or_build_index(video_path, force_reindex=FORCE_REINDEX)

print("Loaded indexes:", list(indexes_by_video.keys()))

Indexing gazelle.mp4 -> gazelle.index.json
Indexing paris.mp4 -> paris.index.json
Indexing waterfall.mp4 -> waterfall.index.json
Loaded indexes: ['gazelle', 'paris', 'waterfall']


## 7. Validate index structure and persistence

This cell checks that each saved index contains the expected assistant artifacts.

In [7]:
validation_rows = []

for video_id, index in indexes_by_video.items():
    persisted_copy = store.load(index_output_path(Path(index.video_path)))
    visual_chunk_count = sum(1 for chunk in index.chunks if chunk.modality == "visual")
    audio_chunk_count = sum(1 for chunk in index.chunks if chunk.modality == "audio")

    validation_rows.append(
        {
            "video": Path(index.video_path).name,
            "summary_present": bool(index.summary.strip()),
            "transcript_segments": len(index.transcript_segments),
            "audio_chunks": audio_chunk_count,
            "visual_chunks": visual_chunk_count,
            "chunk_embeddings_present": all(bool(chunk.embedding) for chunk in index.chunks),
            "round_trip_summary_match": index.summary == persisted_copy.summary,
        }
    )

validation_rows

[{'video': 'gazelle.mp4',
  'summary_present': True,
  'transcript_segments': 7,
  'audio_chunks': 7,
  'visual_chunks': 69,
  'chunk_embeddings_present': True,
  'round_trip_summary_match': True},
 {'video': 'paris.mp4',
  'summary_present': True,
  'transcript_segments': 0,
  'audio_chunks': 0,
  'visual_chunks': 45,
  'chunk_embeddings_present': True,
  'round_trip_summary_match': True},
 {'video': 'waterfall.mp4',
  'summary_present': True,
  'transcript_segments': 0,
  'audio_chunks': 0,
  'visual_chunks': 6,
  'chunk_embeddings_present': True,
  'round_trip_summary_match': True}]

## 8. Inspect summaries and transcript previews

Use this cell to quickly understand what the assistant has indexed before running searches.

In [8]:
def transcript_preview(index: VideoIndex, max_chars: int = 500) -> str:
    text = " ".join(segment.text.strip() for segment in index.transcript_segments if segment.text.strip())
    if not text:
        return "<empty transcript>"
    return text if len(text) <= max_chars else text[:max_chars] + "..."


for video_id, index in indexes_by_video.items():
    print("=" * 100)
    print(f"VIDEO     : {Path(index.video_path).name}")
    print(f"SUMMARY   : {index.summary or '<empty summary>'}")
    print(f"TRANSCRIPT: {transcript_preview(index)}")
    print()

VIDEO     : gazelle.mp4
SUMMARY   : **Summary:**  
The video features a series of high-speed predator-prey interactions in a grassy savanna, showcasing animals like cheetahs, lions, gazelles, and antelopes. Key scenes include a cheetah chasing a deer, a lioness pursuing a horned gazelle, and a cheetah attacking an antelope. The footage highlights the intensity of these chases, with animals kicking up dust as they move rapidly. The National Geographic Wild logo is visible in several frames, indicating the source.  

**Search Keywords:**  
- Cheetah hunting antelope  
- Lioness chasing gazelle  
- Predator-prey interaction in savanna  
- National Geographic Wild wildlife footage  
- Gazelle with horns running from predator  
- Dust kicked up during animal chase  

This video captures the raw dynamics of wildlife survival, ideal for searches on animal behavior, hunting strategies, or National Geographic content.
TRANSCRIPT: अब। Laproit Resistant The corn de soisson Le female ton duriste e

## 9. Run the automated feature checks

For each selected video, this cell tests:

- summary presence
- transcript-oriented search, when transcript exists
- visual-oriented search
- question answering

The goal is coverage, not brittle pass/fail assertions against model output wording.

In [9]:
feature_report = []

for video_id, index in indexes_by_video.items():
    defaults = VIDEO_TEST_CASES.get(video_id, {})
    audio_query = first_audio_query(index)
    visual_query = defaults.get("visual_query", video_id.replace("_", " "))
    qa_question = defaults.get("question", "What happens in this video?")

    audio_results = searcher.search(index, audio_query, top_k=TOP_K) if audio_query else []
    visual_results = searcher.search(index, visual_query, top_k=TOP_K)
    answer = assistant.answer(index, qa_question)

    feature_report.append(
        {
            "video": Path(index.video_path).name,
            "audio_query": audio_query,
            "audio_results_found": len(audio_results),
            "visual_query": visual_query,
            "visual_results_found": len(visual_results),
            "qa_question": qa_question,
            "qa_answer": answer.answer,
            "top_visual_result": result_to_dict(visual_results[0]) if visual_results else None,
            "top_audio_result": result_to_dict(audio_results[0]) if audio_results else None,
        }
    )

feature_report

[{'video': 'gazelle.mp4',
  'audio_query': 'अब।',
  'audio_results_found': 5,
  'visual_query': 'gazelle',
  'visual_results_found': 5,
  'qa_question': 'When does the gazelle appear?',
  'qa_answer': 'The gazelle appears at multiple timestamps in the video. Key moments include:  \n- **0.00s**: A gazelle is shown running in the wild, kicking up dust.  \n- **16.00s**: A cheetah is chasing a gazelle across a grassy plain.  \n- **30.00s**: A gazelle with long, curved horns is running in a savannah-like environment.  \n- **114.00s** and **116.00s**: Close-up frames of a gazelle in mid-stride, highlighting its horns and movement.  \n\nThe gazelle is a recurring subject throughout the video, featured in predator-prey interactions and standalone scenes.',
  'top_visual_result': {'rank': 1,
   'modality': 'visual',
   'label': 'frame @ 0.00s',
   'time_range': '0.00s -> 0.00s',
   'semantic_score': 0.3858,
   'rerank_score': 7.3945,
   'final_score': 7.3945,
   'text': 'This is a video frame o

## 10. Focused search examples

This cell prints the top search results in a more readable way for each default visual query.

In [10]:
for video_id, index in indexes_by_video.items():
    query = VIDEO_TEST_CASES.get(video_id, {}).get("visual_query", video_id)
    results = searcher.search(index, query, top_k=TOP_K)
    print("=" * 100)
    print(f"VIDEO : {Path(index.video_path).name}")
    print(f"QUERY : {query}")
    if not results:
        print("No results found.")
        continue
    for result in results:
        print("-" * 100)
        print(json.dumps(result_to_dict(result), indent=2))
    print()

VIDEO : gazelle.mp4
QUERY : gazelle
----------------------------------------------------------------------------------------------------
{
  "rank": 1,
  "modality": "visual",
  "label": "frame @ 0.00s",
  "time_range": "0.00s -> 0.00s",
  "semantic_score": 0.3858,
  "rerank_score": 7.3945,
  "final_score": 7.3945,
  "text": "This is a video frame of a gazelle running in the wild. The gazelle is in mid-stride, kicking up dust as it runs. The background is blurred, but it appears to be a grassy field with trees in the distance. There is no visible text or landmarks in the frame. The gazelle is the only animal visible in the frame."
}
----------------------------------------------------------------------------------------------------
{
  "rank": 2,
  "modality": "visual",
  "label": "frame @ 30.00s",
  "time_range": "30.00s -> 30.00s",
  "semantic_score": 0.3815,
  "rerank_score": 7.3945,
  "final_score": 7.3945,
  "text": "In this frame from a video, a gazelle is seen running across a g

## 11. Focused QA examples

This cell runs one QA example per video using the default question map.

In [11]:
for video_id, index in indexes_by_video.items():
    question = VIDEO_TEST_CASES.get(video_id, {}).get("question", "What happens in this video?")
    answer = assistant.answer(index, question)
    print("=" * 100)
    print(f"VIDEO    : {Path(index.video_path).name}")
    print(f"QUESTION : {question}")
    print(f"ANSWER   : {answer.answer}")
    print("EVIDENCE :")
    for item in answer.evidence:
        print(f"  - [{item.modality}] {item.start_seconds:.2f}s -> {item.end_seconds:.2f}s | {item.label or item.chunk_id}")
    print()

VIDEO    : gazelle.mp4
QUESTION : When does the gazelle appear?
ANSWER   : The gazelle appears at multiple timestamps in the video. Key moments include:  
- **0.00s**: A gazelle is seen running, kicking up dust in a grassy field.  
- **16.00s**: A cheetah chases a gazelle across a grassy plain, with the gazelle running at full speed.  
- **30.00s**: A gazelle with long, curved horns runs across a grassy field, with trees and bushes in the background.  
- **114.00s**: A gazelle in mid-stride, with a light brown coat and white underbelly, is captured in motion.  
- **116.00s**: A gazelle with long, curved horns is running across a grassy field, with a blurred green background.  

The gazelle is a central figure in several predator-prey interactions throughout the video.
EVIDENCE :
  - [visual] 10.00s -> 10.00s | frame @ 10.00s
  - [visual] 116.00s -> 116.00s | frame @ 116.00s
  - [visual] 114.00s -> 114.00s | frame @ 114.00s
  - [visual] 0.00s -> 0.00s | frame @ 0.00s
  - [visual] 16.00s

## 12. Manual search playground

Use this for ad hoc testing of the search feature on one video.

In [12]:
PLAYGROUND_VIDEO = list(indexes_by_video.keys())[0]
PLAYGROUND_QUERY = "When does the main subject appear?"

playground_index = indexes_by_video[PLAYGROUND_VIDEO]
playground_results = searcher.search(playground_index, PLAYGROUND_QUERY, top_k=TOP_K)

print(f"VIDEO : {Path(playground_index.video_path).name}")
print(f"QUERY : {PLAYGROUND_QUERY}")

for result in playground_results:
    print("-" * 100)
    print(json.dumps(result_to_dict(result), indent=2))

VIDEO : gazelle.mp4
QUERY : When does the main subject appear?
----------------------------------------------------------------------------------------------------
{
  "rank": 1,
  "modality": "visual",
  "label": "frame @ 92.00s",
  "time_range": "92.00s -> 92.00s",
  "semantic_score": 0.1073,
  "rerank_score": -5.6875,
  "final_score": -5.6875,
  "text": "This image is a frame from a video titled \"NATIONAL GEOGRAPHIC WILD.\" It captures a moment in the wild where a lion is seen in the act of hunting or feeding on prey in a grassy savanna. The background is composed of green grass and trees, indicative of an African savanna. The lion is the main subject, with its body positioned over the prey, which is not clearly visible in this frame. The video's title is displayed in the top right corner, with the National Geographic logo and the word \"WILD\" in bold, white letters."
}
----------------------------------------------------------------------------------------------------
{
  "rank":

## 13. Manual QA playground

Use this for free-form chatbot-like testing on one indexed video.

In [13]:
PLAYGROUND_QUESTION = "What happens in this video, and when is the most important visual moment?"

playground_answer = assistant.answer(playground_index, PLAYGROUND_QUESTION)

print(f"VIDEO    : {Path(playground_index.video_path).name}")
print(f"QUESTION : {PLAYGROUND_QUESTION}")
print(f"ANSWER   : {playground_answer.answer}")
print("EVIDENCE :")
for item in playground_answer.evidence:
    print(f"  - [{item.modality}] {item.start_seconds:.2f}s -> {item.end_seconds:.2f}s | {item.label or item.chunk_id}")

VIDEO    : gazelle.mp4
QUESTION : What happens in this video, and when is the most important visual moment?
ANSWER   : The video showcases high-speed predator-prey interactions in a savanna, including a cheetah chasing a deer, a lioness pursuing a horned gazelle, and a cheetah attacking an antelope. Key moments include a cheetah attacking prey with visible claws and a blue light (possibly blood) at **96.00s**, a gazelle running with horns at **114.00s**, and a deer with horns fleeing at **118.00s**. The most critical visual moment is at **96.00s**, where the predator’s attack is dramatized with vivid details like extended claws and the blue light, emphasizing the intensity of the chase. The National Geographic Wild logo appears in multiple frames, reinforcing the source. 

**Answer:** The video depicts predator-prey chases in a savanna, with the most important moment at **96.00s** showing a predator attacking prey with claws and a blue light.
EVIDENCE :
  - [visual] 118.00s -> 118.00s 